# 1.9 梯度代表多敏感？


[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html) 給了衡量猜法好壞的代價。接下來要調參數，卻有一個問題：目前代價高，並不直接告訴我們哪個數字應該增加、哪個減少。先離開文字表，用一個旋鈕看清「附近的變化」。假設旋鈕 w 的理想位置是 3，代價為 `(w-3)²`。目前 w=1，代價是 4；把它稍微增大，會靠近 3，代價應下降。

敏感度用「代價改變多少，除以旋鈕改變多少」來量。若 w 增加 0.1，代價從 4 變成 3.61，比值是 -3.9；若增加 0.01，比值接近 -3.99。當改變數趨近零，這個比例趨近 -4，這叫導數。對多個參數各計一個導數，排列起來便叫梯度；[W.6](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6) 用自動求導介紹同一個概念，本節先用普通數值自己估算。

可以同時往左與往右試：令 h 是一個很小的距離，取 w+h 與 w-h 的代價差，再除以兩點距離 2h。這叫有限差分（finite difference），因為用有限的小步取代無限靠近。它常用來核對自動求導，但不是大型模型通常用的訓練方法；每個參數都這樣試會很慢。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
def loss(w):
    return (w - 3) ** 2


w = 1.0
h = 0.001
left = loss(w - h)
right = loss(w + h)
gradient = (right - left) / (2 * h)
print(left, right)
print(gradient, "公式計算", 2 * (w - 3))

輸入位置 w=1 與距離 h=0.001。`def loss(w)` 定義一個做法，每次提供 w 就回傳代價；`** 2` 是平方。左點 0.999 的代價約 4.004001，右點 1.001 的代價約 3.996001，兩者相差 -0.008，再除以 0.002，得到約 -4。平方函式的精確導數為 `2*(w-3)`，代入同樣位置也得到 -4，所以這次估算能和公式相互核對。

負號的意思是「在目前附近，增加 w 會降低代價」，不是叫我們把 w 改成負數。若 w=5，同樣計算得到正 4，表示再增加會遠離目標，應反方向走。若 w=3，導數是零，當前位置正好在谷底。這些都是區域性資訊，不能據此說任何大小的一步都能改善代價。

h 也有取捨：太大可能跨過曲線變化，算出來不再代表附近。電腦通常用有限位數近似保存小數，這種數字表示叫浮點數；近似值與理想值之間的小差距叫捨入誤差。若 h 太小，左右代價幾乎相同，相減得到的差值本來就很小，兩個近似值的誤差可能在其中佔很大比例。再除以很小的 2h，這個偏差也會被放大，讓梯度估算不準。本例平方函式的左右差分恰好很準，不能用它證明所有函式都不怕大步子。

練習只把 w 改成 5，先預測左右哪個代價小、梯度應是什麼符號，再執行。應看到左點較好、梯度約 +4。接著試 w=3：第一行的左右代價都約為 `0.000001`，因為測量的是 2.999 與 3.001，並非谷底 3 本身。畫面可能把它們寫成 `9.999999999997797e-07`；這種記法中的 `e-07` 表示前面的數乘上 `0.0000001`，所以約是 10 乘 `0.0000001`，也就是 `0.000001`。第二行才是梯度，應看到 `0.0 公式計算 0.0`：左右代價相等，兩者相減為零。這次不需要真正更新參數，只要能把數字的符號翻譯成「往哪邊微調比較好」。
